# Secure Federated Analytics

This notebook demonstrates how to use secure aggregation with federated analytics in Fed-BioMed. Secure aggregation ensures that individual node statistics are encrypted and only the global aggregated result is visible to the researcher.

## Prerequisites

Before running this notebook, ensure you have:
1. At least 2 nodes running with datasets
2. Nodes configured with `allow_federated_analytics = True` (default)
3. For Joye-Libert scheme, proper certificate configuration

## Setting up the nodes

Start your nodes as usual. For this tutorial, we assume you have 2 nodes running with ADNI datasets.

In [ ]:
# First, let's import the required modules
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.secagg import SecureAggregation, SecureAggregationSchemes

## Basic Usage with Secure Aggregation

Enable secure aggregation when creating an experiment using the `secagg` parameter.

In [ ]:
# Create experiment with secure aggregation (default LOM scheme)
exp = Experiment(
    tags=['adni'],  # Use your dataset tags
    secagg=True    # Enable secure aggregation
)

print(f"SecAgg enabled: {exp.analytics.secagg.active}")

## Compute Statistics with Encryption

When computing statistics with secure aggregation enabled, the following happens:
1. The researcher sets up a secure aggregation context with nodes
2. Each node encrypts its statistics before sending
3. Encrypted values are aggregated (summing cancels out masking)
4. Only the global aggregated result is decrypted on the researcher side

In [ ]:
# Compute mean - this will be encrypted and securely aggregated
result = exp.analytics.mean(dataset_args={'col_names': ['AGE']})

# The global result is automatically decrypted
print(f"Global mean AGE: {result.global_stat('mean')}")

## Using Different Schemes

Fed-BioMed supports two secure aggregation schemes:
- **LOM** (Learning with Optical Multiphoton) - default, no extra configuration
- **Joye-Libert** - requires certificate configuration

In [ ]:
# Using LOM scheme (default)
exp_lom = Experiment(tags=['adni'], secagg=True)  # Default is LOM

# Using Joye-Libert scheme (requires certificate configuration)
# secagg = SecureAggregation(scheme=SecureAggregationSchemes.JOYE_LIBERT, active=True)
# exp_jl = Experiment(tags=['adni'], secagg=secagg)

## Multiple Statistics

You can compute multiple statistics at once with secure aggregation.

In [ ]:
# Compute multiple statistics
result = exp.analytics.compute_analytics(
    stats=['mean', 'variance', 'count', 'min', 'max'],
    dataset_args={'col_names': ['AGE']}
)

# Access global aggregated statistics
global_stats = result.global_stats()
print("Global statistics for AGE:")
for stat_name, value in global_stats.get('AGE', {}).items():
    print(f"  {stat_name}: {value}")

## Histogram with Secure Aggregation

For histograms, bin edges are sent in clear (needed for interpretation), but counts are encrypted.

In [ ]:
# Compute histogram securely
result = exp.analytics.compute_analytics(
    stats=['histogram'],
    dataset_args={
        'col_names': ['AGE'],
        'histogram_args': {'bins': 10, 'range': (50, 100)}
    }
)

hist = result.global_stat('histogram')
print(f"Histogram bin_edges: {hist.get('AGE', {}).get('bin_edges', [])}")
print(f"Histogram counts: {hist.get('AGE', {}).get('counts', [])}")

## Dynamic Configuration

You can enable or disable secure aggregation dynamically.

In [ ]:
# Create experiment without secagg
exp_no_sec = Experiment(tags=['adni'], secagg=False)

# Enable secagg later
exp_no_sec.analytics.set_secagg(True)

print(f"SecAgg enabled: {exp_no_sec.analytics.secagg.active}")

## Comparison: With vs Without SecAgg

Let's compare the behavior with and without secure aggregation.

In [ ]:
# Without SecAgg - can access individual node results
exp_no_sec = Experiment(tags=['adni'], secagg=False)
result_no_sec = exp_no_sec.analytics.mean(dataset_args={'col_names': ['AGE']})

print("Without Secure Aggregation:")
print("  Per-node results are accessible:")
for node_id in result_no_sec.node_ids:
    print(f"    Node {node_id}: {result_no_sec.node_stats(node_id)}")

# With SecAgg - only global result is visible
exp_sec = Experiment(tags=['adni'], secagg=True)
result_sec = exp_sec.analytics.mean(dataset_args={'col_names': ['AGE']})

print("\nWith Secure Aggregation:")
print(f"  Global result: {result_sec.global_stat('mean')}")
print("  Individual node results are encrypted and cannot be accessed")

## Supported Statistics

| Statistic | Protection | Notes |
|-----------|------------|-------|
| mean | ✅ Full | Scalar value encrypted |
| variance | ✅ Full | Scalar value encrypted |
| count | ✅ Full | Scalar value encrypted |
| min | ✅ Full | Scalar value encrypted |
| max | ✅ Full | Scalar value encrypted |
| histogram | ⚠️ Partial | Bin edges in clear, counts encrypted |
| quantile | ⚠️ Partial | Derived from histogram |
| shape | ❌ Not supported | Image metadata only |

## Security Properties

### What is Protected
- Individual node statistics are encrypted using the selected scheme (LOM or Joye-Libert)
- Encryption keys are shared among all participants
- Only the sum of encrypted values can be decrypted (not individual contributions)

### What is NOT Protected
- Histogram bin edges: Must be shared in clear for interpretation
- Metadata: Node selection, timing information
- Dataset schema: Column names and types

## See Also

- [Secure Aggregation Tutorial](./secure-aggregation.ipynb)
- [Secure Aggregation User Guide](../user-guide/secagg/introduction.md)
- [LOM Scheme Details](../user-guide/secagg/lom.md)
- [Joye-Libert Scheme Details](../user-guide/secagg/joye-libert.md)